# CrossHair — All Outputs from All Subcommands
Install: `pip install crosshair-tool`

CrossHair has **5 subcommands**, each producing distinct output:

| Subcommand | What it emits |
|------------|---------------|
| `check` | Counterexample lines + return code |
| `cover` | Argument dicts/expressions that cover code paths |
| `search` | Arguments that make a function complete without error |
| `diffbehavior` | Arguments that produce different results between two functions |
| `watch` | Continuous version of `check` (same output format) |

In [1]:
import crosshair
print(dir(crosshair))

['IgnoreAttempt', 'NoTracing', 'ResumedTracing', 'StateSpace', 'SymbolicFactory', '__all__', '__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__license__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__status__', '__version__', 'auditwall', 'codeconfig', 'condition_parser', 'copyext', 'core', 'debug', 'deep_realize', 'dynamic_typing', 'enforce', 'env_info', 'fnutil', 'options', 'patch_to_return', 'realize', 'register_contract', 'register_patch', 'register_type', 'smtlib', 'statespace', 'stubs_parser', 'sys', 'tracers', 'type_repo', 'util', 'with_realized_args', 'z3util']


In [2]:
# Raw AnalysisOptions defaults — all configuration fields
from crosshair.options import DEFAULT_OPTIONS
import dataclasses, json

print(json.dumps(dataclasses.asdict(DEFAULT_OPTIONS), default=str, indent=2))

{
  "analysis_kind": [
    "AnalysisKind.PEP316",
    "AnalysisKind.icontract",
    "AnalysisKind.deal"
  ],
  "enabled": true,
  "specs_complete": false,
  "per_condition_timeout": Infinity,
  "max_iterations": 9223372036854775807,
  "report_all": false,
  "report_verbose": true,
  "unblock": [],
  "timeout": Infinity,
  "per_path_timeout": NaN,
  "max_uninteresting_iterations": 9223372036854775807,
  "deadline": NaN,
  "stats": null
}


---
## `crosshair check` — Contract Verification

**CrossHair does NOT emit JSON.** Its entire output is:

1. **stdout** — zero or more plain text lines, one per counterexample:
```
<filepath>:<lineno>: error: <condition_result> when calling <fn(args)> (which returns <value>)
```
2. **Return code** — `0` (verified), `1` (counterexample found), `2` (other error)

That is all CrossHair natively produces. Nothing else.

**With `--report_verbose`** the same counterexample line is followed by a human-readable source snippet block on stdout (not machine-parseable).
**With `--report_all`** passing conditions are also printed (as `Not confirmed.` blocks).

In [3]:
# check — raw stdout (machine-readable counterexample lines)
import subprocess, os

test_file = os.path.join(os.path.dirname(os.getcwd()), 'crosshair_test.py')

r = subprocess.run(
    ['crosshair', 'check', '--analysis_kind', 'PEP316', '--per_condition_timeout', '5', test_file],
    capture_output=True, text=True, timeout=120
)
print('=== STDOUT (counterexample lines) ===')
print(r.stdout if r.stdout else '(empty — no counterexamples)')
print('=== STDERR ===')
print(r.stderr if r.stderr else '(empty)')
print('=== RETURN CODE ===')
print(r.returncode, '# 0=verified, 1=counterexample found, 2=error')

=== STDOUT (counterexample lines) ===
F:\testable-whitebox-metrics\crosshair_test.py:11: error: false when calling buggy(0) (which returns -1)

=== STDERR ===
(empty)
=== RETURN CODE ===
1 # 0=verified, 1=counterexample found, 2=error


In [1]:
# check --report_all --report_verbose — all conditions + source context + stack trace
import subprocess, os

test_file = os.path.join(os.path.dirname(os.getcwd()), 'crosshair_test.py')

r = subprocess.run(
    ['crosshair', 'check', '--analysis_kind', 'PEP316',
     '--per_condition_timeout', '5',
     '--report_all',      # emit results for ALL conditions, not just failing
     '--report_verbose',  # include source snippet and context per condition
     test_file],
    capture_output=True, text=True, timeout=120
)
print('=== STDOUT (verbose — all conditions) ===')
print(r.stdout if r.stdout else '(empty)')
print('=== RETURN CODE ===')
print(r.returncode)

=== STDOUT (verbose — all conditions) ===

I wasn't able to find a counterexample.
F:\testable-whitebox-metrics\crosshair_test.py:4:
|def divide(a: int, b: int) -> float:
|    """
|    pre: b != 0
>    post: __return__ == a / b
|    """
|    return a / b
|

Not confirmed.


I was able to make your postcondition return False.
F:\testable-whitebox-metrics\crosshair_test.py:11:
|def buggy(x: int) -> int:
|    """
|    pre: x >= 0
>    post: __return__ >= 0
|    """
|    return x - 1
|

false
when calling buggy(0) (which returns -1)


I wasn't able to find a counterexample.
F:\testable-whitebox-metrics\crosshair_test.py:18:
|def off_by_one(items: list, index: int) -> object:
|    """
|    pre: 0 <= index < len(items)
>    post: __return__ == items[index]
|    """
|    return items[index + 1]

Not confirmed.


=== RETURN CODE ===
1


In [5]:
# ============================================================
# IMPORTANT: CrossHair does NOT emit JSON.
#
# CrossHair's actual native output is plain text on stdout:
#
#   <filepath>:<lineno>: <kind>: <message> when calling <call> (which returns <value>)
#
# The JSON below is constructed HERE by parsing that raw text.
# CrossHair never produces it.
# ============================================================

import re, json

# --- Step 1: show exactly what CrossHair actually emitted ---
print('=== WHAT CROSSHAIR ACTUALLY EMITS (raw stdout) ===')
print(repr(r.stdout))          # raw bytes-as-string, shows \n etc.
print()
print(r.stdout if r.stdout else '(empty — no violations)')
print('RETURN CODE (the only other output):', r.returncode)
print()

# --- Step 2: parse that raw text into fields (this is OUR code, not CrossHair) ---
print('=== PARSED REPRESENTATION (constructed by this notebook, NOT by CrossHair) ===')

LINE_RE = re.compile(r'^(?P<filepath>.+?):(?P<lineno>\d+): (?P<kind>\w+): (?P<message>.+)$')
CALL_RE = re.compile(r'(?P<condition>.+?) when calling (?P<call>.+?) \(which returns (?P<return_value>.+)\)')

findings = []
for line in r.stdout.splitlines():
    m = LINE_RE.match(line.strip())
    if not m:
        continue
    entry = {
        'filepath':           m.group('filepath'),
        'lineno':             int(m.group('lineno')),
        'kind':               m.group('kind'),
        'full_message':       m.group('message'),
    }
    cm = CALL_RE.match(m.group('message'))
    if cm:
        entry['condition']           = cm.group('condition')
        entry['counterexample_call'] = cm.group('call')
        entry['return_value']        = cm.group('return_value')
    findings.append(entry)

print(json.dumps({
    'NOTE':                 'This JSON is produced by regex-parsing CrossHair raw stdout — CrossHair does not emit JSON',
    'raw_stdout':           r.stdout,
    'return_code':          r.returncode,
    'verified':             r.returncode == 0,
    'counterexample_count': len(findings),
    'counterexamples':      findings
}, indent=2))

=== WHAT CROSSHAIR ACTUALLY EMITS (raw stdout) ===
'\n\x1bI wasn\'t able to find a counterexample.\x1b\nF:\\testable-whitebox-metrics\\crosshair_test.py:4:\n|def divide(a: int, b: int) -> float:\n|    """\n|    pre: b != 0\n>\x1b    post: __return__ == a / b\n\x1b|    """\n|    return a / b\n|\n\nNot confirmed.\n\n\n\x1bI was able to make your postcondition return False.\x1b\nF:\\testable-whitebox-metrics\\crosshair_test.py:11:\n|def buggy(x: int) -> int:\n|    """\n|    pre: x >= 0\n>\x1b    post: __return__ >= 0\n\x1b|    """\n|    return x - 1\n|\n\nfalse\nwhen calling buggy(0) (which returns -1)\n\n\n\x1bI wasn\'t able to find a counterexample.\x1b\nF:\\testable-whitebox-metrics\\crosshair_test.py:18:\n|def off_by_one(items: list, index: int) -> object:\n|    """\n|    pre: 0 <= index < len(items)\n>\x1b    post: __return__ == items[index]\n\x1b|    """\n|    return items[index + 1]\n\nNot confirmed.\n\n'


I wasn't able to find a counterexample.
F:\testable-whitebox-metrics\crossh

---
## `crosshair cover` — Test Input Generation

`crosshair cover` does **not** produce coverage percentages or line numbers.  
It produces **concrete argument values** — one set per distinct code path it discovers.

### What it actually emits on stdout

| `--example_output_format` | Native stdout output |
|---------------------------|----------------------|
| `arg_dictionary` | `{"a": 0, "b": -1}` — one JSON dict per line |
| `eval_expression` | `divide(0, -1)` — one call expression per line |
| `pytest` | Full pytest test function stubs with import lines |
| `argument_dictionary` | (deprecated alias for arg_dictionary) |

Each output line = one distinct code path CrossHair found.  
**Number of lines = number of distinct paths discovered.**

Return code is always `0`.

### `--coverage_type`
- `opcode` — covers as many opcodes (similar to branch coverage) as possible — **fewer paths**
- `path`   — covers every possible execution path — **more paths, can be infinite**

### `--verbose` stderr  (internal, not normally used)
Emits per-iteration SMT solver decisions, symbolic variable types, path tree stats (`{CONFIRMED: N}`), and `Realized args: {...}` for each path.

In [6]:
# cover — all four output formats side by side
import subprocess, os, json

test_file = os.path.join(os.path.dirname(os.getcwd()), 'crosshair_test.py')

for fmt in ['arg_dictionary', 'eval_expression', 'pytest']:
    r = subprocess.run(
        ['crosshair', 'cover',
         '--coverage_type', 'opcode',
         '--example_output_format', fmt,
         '--per_condition_timeout', '5',
         test_file],
        capture_output=True, text=True, timeout=120
    )
    lines = [l for l in r.stdout.splitlines() if l.strip()]
    print(f'=== --example_output_format {fmt} ===')
    print(f'RETURN CODE: {r.returncode}')
    print(f'PATHS DISCOVERED: {len([l for l in lines if not l.startswith("from") and not l.startswith("def") and not l.startswith("assert") and not l.startswith("import")])}')
    print('RAW STDOUT:')
    print(r.stdout)
    print()

=== --example_output_format arg_dictionary ===
RETURN CODE: 0
PATHS DISCOVERED: 3
RAW STDOUT:
{"items": ['', '', '', ''], "index": -2}
{"a": 0, "b": 1}
{"x": 0}


=== --example_output_format eval_expression ===
RETURN CODE: 0
PATHS DISCOVERED: 3
RAW STDOUT:
off_by_one(['', '', '', ''], -2)
divide(0, 1)
buggy(0)


=== --example_output_format pytest ===
RETURN CODE: 0
PATHS DISCOVERED: 3
RAW STDOUT:
from crosshair_test import buggy
from crosshair_test import divide
from crosshair_test import off_by_one

def test_off_by_one():
    assert off_by_one(['', '', '', ''], -2) == ''

def test_divide():
    assert divide(0, 1) == 0.0

def test_buggy():
    assert buggy(0) == -1





In [7]:
# cover opcode vs path coverage — show difference in path count
import subprocess, os

test_file = os.path.join(os.path.dirname(os.getcwd()), 'crosshair_test.py')

for cov_type in ['opcode', 'path']:
    r = subprocess.run(
        ['crosshair', 'cover',
         '--coverage_type', cov_type,
         '--example_output_format', 'arg_dictionary',
         '--per_condition_timeout', '5',
         test_file],
        capture_output=True, text=True, timeout=120
    )
    lines = [l for l in r.stdout.splitlines() if l.strip()]
    print(f'=== --coverage_type {cov_type} ===')
    print(f'RETURN CODE: {r.returncode}')
    print(f'PATH COUNT (lines in stdout): {len(lines)}')
    print('RAW STDOUT:')
    print(r.stdout)
    print()

=== --coverage_type opcode ===
RETURN CODE: 0
PATH COUNT (lines in stdout): 3
RAW STDOUT:
{"items": ['', '', '', ''], "index": -2}
{"a": 0, "b": 1}
{"x": 0}


=== --coverage_type path ===
RETURN CODE: 0
PATH COUNT (lines in stdout): 35
RAW STDOUT:
{"items": ['', '', '', ''], "index": -2}
{"items": [], "index": -2}
{"items": [0, '\x00', 0, 0, 0, 0, '', 0, '', '\x00'], "index": 7}
{"a": 0, "b": 1}
{"a": 0, "b": 0}
{"a": 0, "b": 0}
{"a": 0, "b": 0}
{"a": 0, "b": 1}
{"a": 0, "b": 2}
{"a": 0, "b": 2}
{"a": 0, "b": 3}
{"a": 0, "b": 3}
{"a": 0, "b": 4}
{"a": 0, "b": 4}
{"a": 0, "b": 5}
{"a": 0, "b": 5}
{"a": 0, "b": 6}
{"a": 0, "b": 6}
{"a": 0, "b": -1}
{"a": 0, "b": -1}
{"a": 0, "b": -1}
{"a": 0, "b": -2}
{"a": 0, "b": -2}
{"a": 0, "b": -3}
{"a": 0, "b": -3}
{"a": 0, "b": -4}
{"a": 0, "b": -4}
{"a": 0, "b": -5}
{"a": 0, "b": -1}
{"a": 0, "b": -5}
{"a": 0, "b": -6}
{"a": 0, "b": -6}
{"a": 0, "b": -7}
{"a": 0, "b": -7}
{"x": 0}




---
## `crosshair search` — Find Valid Function Inputs

**stdout format:** A repr'd dict mapping argument names to values that allow the function to complete without error.

In [8]:
# search — find inputs that make each function complete without error
import subprocess, sys, os

test_file = os.path.join(os.path.dirname(os.getcwd()), 'crosshair_test.py')
sys.path.insert(0, os.path.dirname(test_file))

for fn in ['crosshair_test.divide', 'crosshair_test.buggy', 'crosshair_test.off_by_one']:
    r = subprocess.run(
        ['crosshair', 'search', '--per_condition_timeout', '5',
         '--optimization', 'simplify', fn],
        capture_output=True, text=True, timeout=60,
        cwd=os.path.dirname(test_file)
    )
    print(f'=== search: {fn} ===')
    print('STDOUT:', r.stdout if r.stdout else '(no valid inputs found)')
    print('RETURN CODE:', r.returncode)

=== search: crosshair_test.divide ===
STDOUT: {"a": 0, "b": 1}

RETURN CODE: 0
=== search: crosshair_test.buggy ===
STDOUT: {"x": 0}

RETURN CODE: 0
=== search: crosshair_test.off_by_one ===
STDOUT: {"items": ['', 0], "index": 0}

RETURN CODE: 0


---
## `crosshair diffbehavior` — Behavioral Difference Detection

**stdout format:** Arguments (as a dict) that cause the two functions to return different results or raise different exceptions. Empty stdout = no difference found.

In [9]:
# diffbehavior — find inputs where two function implementations differ
import subprocess, os, tempfile

# Write two functions with different behavior
code = (
    'def impl_v1(x: int, y: int) -> int:\n'
    '    return x + y\n'
    '\n'
    'def impl_v2(x: int, y: int) -> int:\n'
    '    if x == 0:\n'
    '        return y\n'
    '    return x + y + 1  # off by one bug\n'
)

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False,
                                  dir=os.path.dirname(os.getcwd())) as f:
    f.write(code)
    tmp = f.name
    module = os.path.splitext(os.path.basename(tmp))[0]

r = subprocess.run(
    ['crosshair', 'diffbehavior',
     '--per_condition_timeout', '5',
     f'{module}.impl_v1', f'{module}.impl_v2'],
    capture_output=True, text=True, timeout=60,
    cwd=os.path.dirname(tmp)
)
print('=== STDOUT (differentiating inputs) ===')
print(r.stdout if r.stdout else '(no difference found)')
print('=== STDERR ===')
print(r.stderr if r.stderr else '(empty)')
print('=== RETURN CODE ===')
print(r.returncode)
os.unlink(tmp)

=== STDOUT (differentiating inputs) ===
Given: (x=1, y=0),
  tmpmeck96um.impl_v1 : returns 1
  tmpmeck96um.impl_v2 : returns 2

=== STDERR ===
(empty)
=== RETURN CODE ===
1
